In [1]:
## SETUP & LOAD
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sqlalchemy import create_engine
import sys, os

sys.path.insert(0, os.path.abspath('..'))
import config

engine = create_engine(
    f"mysql+pymysql://{config.DB_CONFIG['user']}:{config.DB_CONFIG['password']}"
    f"@{config.DB_CONFIG['host']}:{config.DB_CONFIG['port']}/{config.DB_CONFIG['database']}"
)

os.makedirs('../visuals', exist_ok=True)

vol = pd.read_sql("SELECT * FROM stock_volatility",   engine)
rr  = pd.read_sql("SELECT * FROM stock_risk_return",  engine)
st  = pd.read_sql("SELECT * FROM stocks",             engine)

print("Data loaded.")

Data loaded.


In [2]:
## CHART 1: TOP 15 MOST VOLATILE
top15 = vol.nlargest(15, 'ann_volatility_pct').sort_values('ann_volatility_pct')

fig1 = px.bar(
    top15,
    x='ann_volatility_pct', y='ticker',
    color='market',
    orientation='h',
    text='ann_volatility_pct',
    color_discrete_map={'India': '#EF553B', 'US': '#636EFA'},
    labels={'ann_volatility_pct': 'Annualised Volatility (%)', 'ticker': ''},
    title='Top 15 Most Volatile Stocks — India & US (2010–2024)'
)
fig1.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig1.update_layout(height=550, xaxis_title='Annualised Volatility (%)',
                   legend_title='Market', plot_bgcolor='white')
fig1.write_html('../visuals/01_top_volatile.html')
fig1.show()
print("Saved → visuals/01_top_volatile.html")

Saved → visuals/01_top_volatile.html


In [5]:
## CHART 2: RISK-RETURN SCATTER
rr['drawdown_size'] = rr['max_drawdown_pct'].abs()   # ← convert negatives to positive

fig2 = px.scatter(
    rr,
    x='ann_volatility_pct', y='cagr_pct',
    color='market', size='drawdown_size',             # ← use this instead
    size_max=25,
    hover_data=['ticker', 'company_name', 'sector', 'sharpe_ratio'],
    color_discrete_map={'India': '#EF553B', 'US': '#636EFA'},
    labels={
        'ann_volatility_pct': 'Annualised Volatility (%)',
        'cagr_pct':           'CAGR (%)',
        'drawdown_size':      'Max Drawdown (%)'
    },
    title='Risk vs Return — All 79 Stocks (bubble size = max drawdown)'
)
fig2.update_layout(height=550, plot_bgcolor='white')
fig2.write_html('../visuals/02_risk_return_scatter.html')
fig2.show()
print("Saved → visuals/02_risk_return_scatter.html")

Saved → visuals/02_risk_return_scatter.html


In [6]:
## CHART 3: SECTOR VOLATILITY — INDIA vs US
sector_vol = (vol.groupby(['market', 'sector'])['ann_volatility_pct']
                 .mean().round(2).reset_index())

fig3 = px.bar(
    sector_vol,
    x='sector', y='ann_volatility_pct',
    color='market', barmode='group',
    color_discrete_map={'India': '#EF553B', 'US': '#636EFA'},
    labels={'ann_volatility_pct': 'Avg Annualised Volatility (%)', 'sector': 'Sector'},
    title='Sector-wise Volatility — India vs US'
)
fig3.update_layout(height=480, xaxis_tickangle=-30, plot_bgcolor='white')
fig3.write_html('../visuals/03_sector_volatility.html')
fig3.show()
print("Saved → visuals/03_sector_volatility.html")

Saved → visuals/03_sector_volatility.html


In [7]:
## CHART 4: CORRELATION HEATMAP
for market in ['India', 'US']:
    corr_path = f'../data/processed/corr_{market.lower()}.csv'
    corr = pd.read_csv(corr_path, index_col=0)

    fig = px.imshow(
        corr,
        color_continuous_scale='RdBu_r',
        zmin=-1, zmax=1,
        title=f'Return Correlation Heatmap — {market} Stocks',
        aspect='auto'
    )
    fig.update_layout(height=650)
    fig.write_html(f'../visuals/04_corr_{market.lower()}.html')
    fig.show()
    print(f"Saved → visuals/04_corr_{market.lower()}.html")

Saved → visuals/04_corr_india.html


Saved → visuals/04_corr_us.html


In [8]:
## CHART 5: ANNUAL RETURNS HEATMAP (India)
from sqlalchemy import text

q = """
WITH year_bounds AS (
    SELECT ticker, YEAR(trade_date) AS yr,
           MIN(trade_date) AS fd, MAX(trade_date) AS ld
    FROM daily_prices GROUP BY ticker, YEAR(trade_date)
)
SELECT yb.ticker,
       yb.yr AS year,
       ROUND(((lp.close_price-fp.close_price)/fp.close_price)*100,2) AS annual_return_pct
FROM year_bounds yb
JOIN daily_prices fp ON yb.ticker=fp.ticker AND fp.trade_date=yb.fd
JOIN daily_prices lp ON yb.ticker=lp.ticker AND lp.trade_date=yb.ld
"""
with engine.connect() as conn:
    annual = pd.DataFrame(conn.execute(text(q)).mappings().all())

for market, tickers in [('India', st[st['market']=='India']['ticker'].tolist()),
                         ('US',    st[st['market']=='US']['ticker'].tolist())]:
    pivot = (annual[annual['ticker'].isin(tickers)]
             .pivot(index='ticker', columns='year', values='annual_return_pct'))

    fig = px.imshow(
        pivot,
        color_continuous_scale='RdYlGn',
        zmin=-60, zmax=100,
        title=f'Annual Returns Heatmap — {market} Stocks (%)',
        labels={'color': 'Annual Return (%)'},
        aspect='auto'
    )
    fig.update_layout(height=700)
    fig.write_html(f'../visuals/05_annual_heatmap_{market.lower()}.html')
    fig.show()
    print(f"Saved → visuals/05_annual_heatmap_{market.lower()}.html")

Saved → visuals/05_annual_heatmap_india.html


Saved → visuals/05_annual_heatmap_us.html


In [9]:
## CHART 6: PRICE TREND + MA LINES — TOP 3 VOLATILE
top3 = vol.nlargest(3, 'ann_volatility_pct')['ticker'].tolist()
print(f"Plotting trend for: {top3}")

sf_trend = pd.read_sql(
    f"SELECT ticker, trade_date, close_price, ma_50, ma_200 "
    f"FROM stock_features "
    f"WHERE ticker IN {tuple(top3)} "
    f"ORDER BY ticker, trade_date",
    engine, parse_dates=['trade_date']
)

for ticker in top3:
    df_t = sf_trend[sf_trend['ticker'] == ticker]

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=df_t['trade_date'], y=df_t['close_price'],
                             name='Close', line=dict(color='#636EFA', width=1)))
    fig.add_trace(go.Scatter(x=df_t['trade_date'], y=df_t['ma_50'],
                             name='MA 50',  line=dict(color='#FFA15A', width=1.5, dash='dot')))
    fig.add_trace(go.Scatter(x=df_t['trade_date'], y=df_t['ma_200'],
                             name='MA 200', line=dict(color='#EF553B', width=1.5, dash='dash')))

    fig.update_layout(
        title=f'{ticker} — Price Trend with MA50 & MA200 (2010–2024)',
        xaxis_title='Date', yaxis_title='Price',
        height=450, plot_bgcolor='white', hovermode='x unified'
    )
    safe = ticker.replace('.', '_').replace('&', 'AND')
    fig.write_html(f'../visuals/06_trend_{safe}.html')
    fig.show()
    print(f"Saved → visuals/06_trend_{safe}.html")

Plotting trend for: ['TSLA', 'AMD', 'ADANIENT.NS']


Saved → visuals/06_trend_TSLA.html


Saved → visuals/06_trend_AMD.html


Saved → visuals/06_trend_ADANIENT_NS.html
